## Data source

Publication metadata and abstracts were collected programmatically from
PubMed using the NCBI Entrez Programming Utilities.

The search was restricted to English-language publications containing an
abstract. Publication years in the final corpus cover recent articles from
2025–2026.

In [11]:
from Bio import Entrez, Medline
import pandas as pd
import time
from pathlib import Path

In [ ]:
Entrez.email = "your_email@example.com"

QUERY = """
("antibiotic resistance"[Title/Abstract])
AND hasabstract
AND english[Language]
AND ("2010/01/01"[Date - Publication] : "2025/12/31"[Date - Publication])
"""


MAX_RECORDS = 1000
BATCH_SIZE = 200

In [39]:
search_handle = Entrez.esearch(
    db="pubmed",
    term=QUERY,
    retmax=MAX_RECORDS,
    sort="pub date"
)

search_result = Entrez.read(search_handle)
search_handle.close()

pmids = search_result["IdList"]

In [35]:
records = []

for start in range(0, len(pmids), BATCH_SIZE):
    batch = pmids[start:start + BATCH_SIZE]

    fetch_handle = Entrez.efetch(
        db="pubmed",
        id=batch,
        rettype="medline",
        retmode="text"
    )

    batch_records = list(Medline.parse(fetch_handle))
    fetch_handle.close()

    records.extend(batch_records)

    print(f"Downloaded: {len(records)} / {len(pmids)}")
    time.sleep(0.34)

Downloaded: 200 / 1000
Downloaded: 400 / 1000
Downloaded: 600 / 1000
Downloaded: 800 / 1000
Downloaded: 1000 / 1000


In [36]:
def extract_year(publication_date):
    if not publication_date:
        return None

    year = str(publication_date)[:4]
    return int(year) if year.isdigit() else None


articles = []

for record in records:
    abstract = record.get("AB", "")
    title = record.get("TI", "")

    if not abstract:
        continue

    articles.append({
        "pmid": record.get("PMID"),
        "title": title,
        "abstract": abstract,
        "publication_year": extract_year(record.get("DP")),
        "journal": record.get("JT"),
        "authors": record.get("AU")
    })

df = pd.DataFrame(articles)

df.head()

,pmid,title,abstract,publication_year,journal,authors
0,41090395,Diagnostic accuracy of otitis media with and w...,BACKGROUND: Otitis media (OM) in children is a...,2026,Scandinavian journal of primary health care,"[Hedman M, Kosuta V, Lindmark M, Sandstrom J, ..."
1,42498403,Plastic-mediated alterations in soil microbial...,The spatiotemporal co-accumulation of plastics...,2026,Journal of environmental sciences (China),"[Ye Y, Shen L, Lin D, Zhang T, Wang Y, Lu L, Q..."
2,42498393,Impact of antimicrobial peptide Hidefensin5 as...,Antibiotic resistance has arisen as a formidab...,2026,Journal of environmental sciences (China),"[Xia J, Lu Z, Ge C, Yao H]"
3,42498390,Release and bacterial transformation activity ...,Dissolved sulfides are widely distributed in a...,2026,Journal of environmental sciences (China),"[Yi L, Zhang W, Li H, Liu J, Zhang Z, Lu Y, Zh..."
4,40892486,Molecular Characterization of beta-Lactamase-R...,The number of dairy farms in Bangladesh is ste...,2026,Foodborne pathogens and disease,"[Fahim FJ, Prome AA, Rana S, Uddin MS, Noor M,..."


In [37]:
df["publication_year"].value_counts().sort_index()

publication_year
2025    368
2026    632
Name: count, dtype: int64

In [19]:
print(f"Articles with abstracts: {len(df)}")
print(f"Duplicate PMIDs: {df['pmid'].duplicated().sum()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
print(f"Missing publication years: {df['publication_year'].isna().sum()}")

Articles with abstracts: 1000
Duplicate PMIDs: 0
Missing abstracts: 0
Missing publication years: 0


In [20]:
print(f"Duplicate abstracts: {df['abstract'].duplicated().sum()}")

Duplicate abstracts: 0


In [10]:
output_dir = Path("../data")
output_dir.mkdir(exist_ok=True)

df.to_csv(
    output_dir / "pubmed_antibiotic_resistance.csv",
    index=False
)

In [38]:
print(f"Rows: {len(df):,}")
print(f"Duplicate PMIDs: {df['pmid'].duplicated().sum()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
print(f"Year range: {df['publication_year'].min()}–{df['publication_year'].max()}")
print(df["publication_year"].value_counts().sort_index())

Rows: 1,000
Duplicate PMIDs: 0
Missing abstracts: 0
Year range: 2025–2026
publication_year
2025    368
2026    632
Name: count, dtype: int64


The search retrieved 1,000 PubMed abstracts with no duplicate PMIDs and no
missing abstracts. The resulting corpus is ready for text preprocessing and
topic modeling.